# IOU Heatmap over Attention Patterns

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


IOU heatmaps are a compact way to compare binary attention masks across heads or conditions. They are especially helpful when a visual attention pattern is noisy but the question is set overlap: which heads attend to the same token pairs above a threshold?


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1602
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyAttention(nn.Module):
    """Two-head self-attention block with exposed softmax attention."""

    def __init__(self, width: int = 8, heads: int = 2) -> None:
        """Initialize attention projections."""
        super().__init__()
        self.width = width
        self.heads = heads
        self.head_width = width // heads
        self.q = nn.Linear(width, width, bias=False)
        self.k = nn.Linear(width, width, bias=False)
        self.v = nn.Linear(width, width, bias=False)
        self.out = nn.Linear(width, width, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run scaled dot-product attention."""
        batch, tokens, _ = x.shape
        q = self.q(x).view(batch, tokens, self.heads, self.head_width).transpose(1, 2)
        k = self.k(x).view(batch, tokens, self.heads, self.head_width).transpose(1, 2)
        v = self.v(x).view(batch, tokens, self.heads, self.head_width).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_width)
        attention = torch.softmax(scores, dim=-1)
        context = torch.matmul(attention, v).transpose(1, 2).reshape(batch, tokens, self.width)
        return self.out(context)


def pairwise_iou(masks: torch.Tensor) -> torch.Tensor:
    """Compute pairwise intersection-over-union for flattened boolean masks."""
    flat = masks.reshape(masks.shape[0], -1).bool()
    out = torch.zeros(flat.shape[0], flat.shape[0])
    for i in range(flat.shape[0]):
        for j in range(flat.shape[0]):
            intersection = torch.logical_and(flat[i], flat[j]).sum()
            union = torch.logical_or(flat[i], flat[j]).sum().clamp_min(1)
            out[i, j] = intersection.float() / union.float()
    return out

In [ ]:
model = TinyAttention().eval()
x = torch.randn(1, 5, 8)
log = tl.log_forward_pass(model, x, vis_opt="none", intervention_ready=True)
attention = log.find_sites(tl.func("softmax")).first().activation.squeeze(0)
print(f"attention tensor shape: {tuple(attention.shape)}")
print(f"head 0 row sums: {attention[0].sum(dim=-1).round(decimals=4)}")

In [ ]:
threshold = 0.20
masks = attention > threshold
iou = pairwise_iou(masks)
print(iou)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
image = ax.imshow(iou.numpy(), vmin=0, vmax=1, cmap="magma")
ax.set_xticks(range(attention.shape[0]))
ax.set_yticks(range(attention.shape[0]))
ax.set_xlabel("head")
ax.set_ylabel("head")
ax.set_title(f"Attention mask IOU, threshold>{threshold}")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

For real attention analysis, the only substitution is the attention site: resolve the model's actual attention-probability layer, collect one mask per head, then reuse the same IOU computation and visualization.
